# ID-only version

In [ ]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pandas as pd

from algorithm_evaluation import evaluate

# The sweeps take hours. Leave False to load the shipped results from
# algorithm_evaluation/results/; set True to regenerate and overwrite them.
RECOMPUTE = False

## bnlearn graphs (ID-only)

In [ ]:
small_networks  = ['asia', 'sachs']
medium_networks = ['alarm', 'barley', 'child', 'insurance', 'mildew', 'water']
large_networks  = ['hailfinder', 'hepar2', 'win95pts']

all_graphs = (
    [(g, 'small-networks')  for g in small_networks]  +
    [(g, 'medium-networks') for g in medium_networks] +
    [(g, 'large-networks')  for g in large_networks]
)

path_prefix = 'algorithm_evaluation/example-graphs/bnlearn/'

theta_config = {
    'CF_fraction': 1.0,
    'W_sizes': [1],
    'Z_sizes': [1],
}

experiment_config = {
    'CF_fraction': 0.3,
    'W_sizes': [1, 2, 3],
    'Z_sizes': [1, 2, 3],
}

N_SIMS   = 100
EXP_SIZE = 200
SEED     = 42

if RECOMPUTE:
    df_bnlearn = pd.DataFrame()

    for graph_name, subfolder in all_graphs:
        bif_path = f"{path_prefix}{subfolder}/{graph_name}.bif"
        print(f"Evaluating {graph_name}.bif ...")
        chunk = evaluate(
            graph_source="bnlearn",
            graph_config={"bif_path": bif_path},
            confounder_ratio_range=(0.1, 0.9),
            confounder_strategy="random",
            n_simulations=N_SIMS,
            seed=SEED,
            verbose=True,
            experiment_set_size=EXP_SIZE,
            theta_config=theta_config,
            experiment_config=experiment_config,
            ID_only=True,
        )
        df_bnlearn = pd.concat([df_bnlearn, chunk], ignore_index=True)

    df_bnlearn.to_csv('algorithm_evaluation/results/bnlearn_ID_only.csv', index=False)
    print(f"\nTotal simulations: {len(df_bnlearn)}")

In [ ]:
# Written above when RECOMPUTE is True; otherwise the shipped results.
df_bnlearn = pd.read_csv('algorithm_evaluation/results/bnlearn_ID_only.csv')

In [ ]:
frac_id_bn = df_bnlearn['id_pruned'] / df_bnlearn['n_experiments']

print("=== bnlearn ID-only — overall ===")
print(f"Total simulations: {len(df_bnlearn)}")
print(f"Mean ID pruning: {frac_id_bn.mean():.3f} (std {frac_id_bn.std():.3f})")
print(f"Max ID pruning (sim): {frac_id_bn.max():.3f}")

per_graph_id = (
    df_bnlearn.assign(frac_id=frac_id_bn)
      .groupby('graph')
      .agg(
          n_vars  = ('n_vars',  'first'),
          n_sims  = ('sim',     'count'),
          uv_mean = ('u_v_ratio', 'mean'),
          id_mean = ('frac_id', 'mean'),
          id_std  = ('frac_id', 'std'),
      )
      .sort_values('n_vars')
      .round(3)
)
print("\nPer-graph ID-only pruning (mean, std):")
print(per_graph_id.to_string())

## Erdős-Rényi sweep (ID-only)

In [ ]:
N_values = [10, 15, 20, 30]
c_values = [0.5, 1.0, 1.5, 2.0]
N_SIMS   = 100
EXP_SIZE = 200
SEED     = 42

theta_config = {
    'CF_fraction': 1.0,
    'W_sizes': [1],
    'Z_sizes': [1],
}

experiment_config = {
    'CF_fraction': 0.3,
    'W_sizes': [1, 2, 3],
    'Z_sizes': [1, 2, 3],
}

if RECOMPUTE:
    rows = []
    for N in N_values:
        p = 0.5 * np.log(N) / N
        for c in c_values:
            q = c / N
            print(f"N={N:2d}, c={c:.1f}  (p={p:.3f}, q={q:.4f}) ...", end=" ", flush=True)
            df_chunk = evaluate(
                graph_source="erdos_renyi",
                graph_config={"N": N, "p": p, "q": q},
                n_simulations=N_SIMS,
                seed=SEED,
                experiment_set_size=EXP_SIZE,
                theta_config=theta_config,
                experiment_config=experiment_config,
                ID_only=True,
            )
            df_chunk["c"] = c
            rows.append(df_chunk)
            total = df_chunk["n_experiments"].sum()
            print(
                f"id-pruned={df_chunk['id_pruned'].sum()/total:.0%}  "
                f"|U|/|V|={df_chunk['u_v_ratio'].mean():.2f}"
            )

    df_er = pd.concat(rows, ignore_index=True)
    df_er.to_csv('algorithm_evaluation/results/er_ID_only.csv', index=False)
    print(f"\nTotal simulations: {len(df_er)}")

In [ ]:
# Written above when RECOMPUTE is True; otherwise the shipped results.
df_er = pd.read_csv('algorithm_evaluation/results/er_ID_only.csv')

In [ ]:
frac_id_er = df_er['id_pruned'] / df_er['n_experiments']

print("=== ER ID-only — overall ===")
print(f"Total simulations: {len(df_er)}")
print(f"Mean ID pruning: {frac_id_er.mean():.3f} (std {frac_id_er.std():.3f})")
print(f"Max ID pruning (sim): {frac_id_er.max():.3f}")

per_c = (
    df_er.assign(frac_id=frac_id_er)
      .groupby(['n_vars', 'c'])
      .agg(
          uv_mean = ('u_v_ratio', 'mean'),
          id_mean = ('frac_id', 'mean'),
          id_std  = ('frac_id', 'std'),
      )
      .round(3)
)
print("\nPer-(N, c) ID-only pruning (mean, std):")
print(per_c.to_string())

per_c_only = (
    df_er.assign(frac_id=frac_id_er)
      .groupby('c')
      .agg(
          uv_mean = ('u_v_ratio', 'mean'),
          id_mean = ('frac_id', 'mean'),
          id_std  = ('frac_id', 'std'),
      )
      .round(3)
)
print("\nPer-c ID-only pruning (mean, std), pooled over N:")
print(per_c_only.to_string())